# Arabic IR Pipeline — Main Notebook

**Improving Arabic Information Retrieval and Reranking Performance using Knowledge Distillation**

Pipeline sections:
1. Configure
2. W&B setup
3. Load data
4. Hard negative mining
5. Train bi-encoder
6. Train cross-encoder
7. First-stage retrieval (FAISS)
8. Reranking
9. **Evaluate** — MRR@10, NDCG@10, Recall@K, MAP@10
10. **Generalization Ratio (GR)** — zero-shot Mr.TyDi vs in-domain mMARCO
11. HTML report
12. Finish W&B
13. (Optional) Multi-model evaluation + GR

## 0. Install dependencies

In [ ]:
import subprocess, sys, torch
from packaging.version import Version
import numpy as _np

# ── Pinned versions known to work together ──────────────────────────────────
VERSIONS = {
    "sentence_transformers": "5.5.0",
    "transformers":          "5.8.1",
    "datasets":              "4.8.5",
    "huggingface_hub":       "1.15.0",
    "accelerate":            "1.13.0",
    "peft":                  "0.19.1",
    "wandb":                 "0.27.0",
    "matplotlib":            "3.10.3",
    "pandas":                "2.3.0",
}

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# ── Select faiss version based on installed numpy ────────────────────────────
# faiss-gpu-cu12/cu11 ==1.14.x  →  requires numpy>=2
# faiss-gpu-cu12/cu11 ==1.9.0   →  requires numpy<2   (safe for numpy 1.26.x)
# faiss-cpu           ==1.9.0   →  requires numpy>=1.25,<3 (works everywhere)
#
# We intentionally do NOT upgrade numpy — scikit-learn / scipy / tensorflow
# in this environment pin numpy<2, so we let numpy stay as-is and pick the
# matching faiss build instead.
_np_new  = Version(_np.__version__) >= Version("2.0")
_faiss_v = "1.14.1.post1" if _np_new else "1.9.0.post1"

_cuda  = torch.version.cuda or ""
_major = int(_cuda.split(".")[0]) if _cuda else 0

if _major >= 12:
    _candidates = [
        f"faiss-gpu-cu12=={_faiss_v}",
        f"faiss-gpu-cu11=={_faiss_v}",
        "faiss-cpu==1.9.0.post1",
    ]
elif _major == 11:
    _candidates = [
        f"faiss-gpu-cu11=={_faiss_v}",
        "faiss-cpu==1.9.0.post1",
    ]
else:
    _candidates = ["faiss-cpu==1.9.0.post1"]

_faiss_installed = None
for _pkg in _candidates:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", _pkg],
        capture_output=True,
    )
    if r.returncode == 0:
        _faiss_installed = _pkg
        break

if _faiss_installed is None:
    raise RuntimeError("Could not install any faiss variant — check your CUDA / pip setup.")

# ── Install remaining pinned dependencies ────────────────────────────────────
_pip(*[f"{k}=={v}" for k, v in VERSIONS.items()])

print(f"numpy version  : {_np.__version__}")
print(f"CUDA version   : {_cuda or 'none'}")
print(f"faiss installed: {_faiss_installed}")
print("All dependencies installed.")

## 1. Configuration

**Edit only this cell** to control the entire pipeline.

| `base_model` key | HuggingFace ID | Role |
|-----------------|---------------|------|
| `"AraELECTRA"` | `aubmindlab/araelectra-base-discriminator` | bi / cross |
| `"AraDPR"` | `abdoelsayed/AraDPR` | bi / cross |
| `"mMiniLML"` | `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` | bi |
| `"mMiniLMv2CE"` | `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` | cross / KD teacher |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from src.config.config import (
    PipelineConfig, DataConfig, ModelConfig,
    TrainingConfig, EvalConfig, LoRAConfig
)

config = PipelineConfig(
    data=DataConfig(
        dataset="mmarco",
        max_train_samples=None,
    ),
    model=ModelConfig(
        base_model="AraDPR",       # "AraELECTRA" | "AraDPR" | "mMiniLML" | "mMiniLMv2CE"
        encoder_type="both",       # "bi" | "cross" | "both"
        max_length=256,
        use_lora=False,
        lora_config=LoRAConfig(r=16, lora_alpha=32, target_modules=None, lora_dropout=0.1),
        use_matryoshka=False,
        matryoshka_dims=[768, 512, 256, 128],
    ),
    training=TrainingConfig(
        use_kd=True,
        kd_mode="listwise",        # "pairwise" | "listwise"
        kd_lambda=1.0,             # λ=1 → pure KD | 0<λ<1 → combined | λ=0 → vanilla fine-tuning
        kd_temperature=1.0,
        hard_negatives=True,
        hn_strategy="curriculum",
        hn_bm25_top_k=200,
        hn_ance_top_k=200,
        hn_refresh_steps=5000,
        curriculum_schedule=[(1, "bm25"), (2, "combined"), (3, "ance")],
        use_mnrl_hybrid=True,
        mnrl_weight=0.5,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=8,
        learning_rate=7e-5,
        weight_decay=0.0,
        warmup_ratio=0.07,
        fp16=True,
        eval_steps=2000,
        output_dir="./training_output",
        report_to=None,            # None | "wandb" | "tensorboard"
        run_name="arabic-ir-kd",
        wandb_project="arabic-ir-kd",
        wandb_entity=None,
        wandb_api_key=None,
        wandb_log_artifacts=True,
    ),
    evaluation=EvalConfig(
        metrics=["mrr@10", "ndcg@10", "recall@10", "recall@100", "recall@1000", "map@10"],
        eval_batch_size=64,
        first_stage_top_k=1000,
        rerank_top_k=1000,
    ),
    report_output="./report.html",
    push_to_hub=False,
    hub_token=None,
    hub_repo_id=None,
    seed=42,
)

print(config.summary())

## 2. Weights & Biases Setup

Set `report_to="wandb"` in the config above to enable. No-op when disabled.

In [ ]:
from src.utils import wandb_logger

wb_run = wandb_logger.init(config)
if wb_run:
    print(f"W&B run: {wb_run.url}  |  project={wb_run.project}  |  name={wb_run.name}")
else:
    print("W&B disabled (set report_to='wandb' to enable)")

## 3. Load Data

In [ ]:
from src.data.loader import DatasetLoader

loader = DatasetLoader(config.data)
queries    = loader.load_queries(split="dev")
corpus     = loader.load_corpus()
dev_samples = loader.load_dev_samples()
qrels      = loader.load_qrels()

print(f"{len(queries)} queries | {len(corpus)} docs | {len(dev_samples)} dev queries | {len(qrels)} qrels")

In [ ]:
train_dataset = loader.load_train_dataset()
print(train_dataset)

## 4. Hard Negative Mining

In [ ]:
from src.data.hard_negatives import HardNegativeMiner
from src.retrieval.bm25_retrieval import BM25Retriever

miner = HardNegativeMiner(
    strategy=config.training.hn_strategy,
    bm25_top_k=config.training.hn_bm25_top_k,
    ance_top_k=config.training.hn_ance_top_k,
    curriculum_schedule=config.training.curriculum_schedule,
    seed=config.seed,
)
miner._queries = queries
miner._corpus  = corpus

bm25_run_mmarco = None
if config.training.hard_negatives:
    bm25_run_path = os.path.join(config.data.cache_dir, "bm25_run.txt")
    if os.path.exists(bm25_run_path):
        bm25 = BM25Retriever()
        bm25_run_mmarco = bm25.load_run(bm25_run_path, k=config.training.hn_bm25_top_k)
        miner.set_bm25_negatives(bm25_run_mmarco, qrels)
        print(f"BM25 negatives ready for {len(miner._bm25_negatives)} queries")
    else:
        print("No cached BM25 run — ANCE only. Use BM25Retriever.retrieve() to generate one.")

## 5. Train Bi-Encoder

In [ ]:
from src.models.bi_encoder import BiEncoderModel
from src.training.bi_encoder_trainer import BiEncoderTrainer

bi_model = None
if config.model.encoder_type in ("bi", "both"):
    bi_model = BiEncoderModel(config.model)
    bi_trainer = BiEncoderTrainer(config, bi_model)
    bi_trainer.train(train_dataset, dev_samples, miner if config.training.hard_negatives else None)
    bi_save_path = os.path.join(config.training.output_dir, "bi_encoder", "final")
    bi_model.save(bi_save_path)
    if config.training.wandb_log_artifacts and config.use_wandb:
        wandb_logger.log_artifact(bi_save_path, "bi-encoder-model", "model")
    if config.push_to_hub and config.hub_repo_id:
        bi_model.push_to_hub(config.hub_repo_id + "-bi", token=config.hub_token)
    print(f"Bi-encoder saved → {bi_save_path}")
else:
    print("Skipping bi-encoder training")

## 6. Train Cross-Encoder

In [ ]:
from src.models.cross_encoder import CrossEncoderModel
from src.training.cross_encoder_trainer import CrossEncoderTrainer

ce_model = None
if config.model.encoder_type in ("cross", "both"):
    ce_model = CrossEncoderModel(config.model)
    ce_trainer = CrossEncoderTrainer(config, ce_model)
    ce_trainer.train(train_dataset, dev_samples)
    ce_save_path = os.path.join(config.training.output_dir, "cross_encoder", "final")
    ce_model.save(ce_save_path)
    if config.training.wandb_log_artifacts and config.use_wandb:
        wandb_logger.log_artifact(ce_save_path, "cross-encoder-model", "model")
    if config.push_to_hub and config.hub_repo_id:
        ce_model.push_to_hub(config.hub_repo_id + "-ce", token=config.hub_token)
    print(f"Cross-encoder saved → {ce_save_path}")
else:
    print("Skipping cross-encoder training")

## 7. First-Stage Retrieval (FAISS)

In [ ]:
from src.retrieval.faiss_retrieval import FaissRetriever

if bi_model is None:
    bi_model = BiEncoderModel.from_pretrained(
        "hatemestinbejaia/mmarco-Arabic-AraDPR-bi-encoder-KD-v1", config.model
    )

retriever = FaissRetriever(bi_encoder=bi_model.get_sentence_transformer(), top_k=1000, batch_size=128)
retriever.build_index(corpus)
first_stage_run = retriever.retrieve(queries)
print(f"Retrieved for {len(first_stage_run)} queries")
retriever.save_run(first_stage_run, os.path.join(config.training.output_dir, "first_stage_run.tsv"))

## 8. Reranking

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder
from src.evaluation.evaluator import PipelineEvaluator

if ce_model is None:
    ce_reranker = CrossEncoder(
        "hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-KD-v1",
        max_length=config.model.max_length
    )
else:
    ce_reranker = CrossEncoder(
        os.path.join(config.training.output_dir, "cross_encoder", "final"),
        max_length=config.model.max_length
    )

# In-domain evaluator (mMARCO)
evaluator_mmarco = PipelineEvaluator(config, queries, corpus, qrels, dataset_name="mmarco")

print("Reranking on mMARCO…")
reranked_run = evaluator_mmarco._rerank(
    reranker=ce_reranker,
    first_stage_run=first_stage_run,
    top_k=config.evaluation.rerank_top_k,
    use_cross_encoder=True,
)
print("Done.")

## 9. Evaluate — In-Domain (mMARCO)

Two evaluation scenarios:
- **Bi-encoder @1000**: FAISS dense retrieval, top-1000 candidates
- **Cross-encoder on BM25**: rerank BM25 top-K candidates (set `bm25_run_mmarco` in section 4)

In [ ]:
import pandas as pd

kd_label    = config.training.kd_mode if config.training.use_kd else "nokd"
model_label = config.model.base_model

# ── Scenario 1: Bi-encoder top-1000 (in-domain mMARCO) ──
bi_mmarco = evaluator_mmarco.evaluate_bi_encoder_top1000(
    bi_encoder=bi_model,
    model_name=model_label,
    kd_mode=kd_label,
)

# ── Scenario 2: Cross-encoder on FAISS top-1000 (in-domain mMARCO) ──
ce_faiss_mmarco = evaluator_mmarco.evaluate_reranking(
    reranker=ce_reranker,
    first_stage_run=first_stage_run,
    model_name=model_label,
    kd_mode=kd_label,
    use_cross_encoder=True,
)

# ── Scenario 3: Cross-encoder on BM25 candidates (in-domain mMARCO) ──
ce_bm25_mmarco = None
if bm25_run_mmarco is not None:
    ce_bm25_mmarco = evaluator_mmarco.evaluate_cross_encoder_on_bm25(
        cross_encoder=ce_reranker,
        bm25_run=bm25_run_mmarco,
        model_name=model_label,
        kd_mode=kd_label,
    )
else:
    print("No BM25 run available for cross-encoder-on-BM25 evaluation.")

indomain_results = [bi_mmarco, ce_faiss_mmarco]
if ce_bm25_mmarco:
    indomain_results.append(ce_bm25_mmarco)

df_indomain = pd.DataFrame([r.as_dict() for r in indomain_results])
print("\n=== In-Domain (mMARCO) ===")
display(df_indomain)

## 10. Generalization Ratio (GR) — Zero-Shot Mr.TyDi

**GR = zero-shot metric / in-domain metric**

- **GR ≥ 1.0** → model generalises fully (or even better) on unseen data
- **GR 0.8–1.0** → partial generalisation
- **GR < 0.8** → poor generalisation

We evaluate the **same trained model** on Mr.TyDi Arabic without any fine-tuning on it (true zero-shot).

In [ ]:
from src.data.loader import DatasetLoader
from src.config.config import DataConfig
from src.evaluation.evaluator import PipelineEvaluator
from src.evaluation.metrics import compute_generalization_ratio

# Load Mr.TyDi data
mrtydi_data_cfg = DataConfig(dataset="mrtydi")
mrtydi_loader   = DatasetLoader(mrtydi_data_cfg)

print("Loading Mr.TyDi queries…")
mrtydi_queries = mrtydi_loader.load_queries(split="test")   # Mr.TyDi uses 'test' split
print(f"  {len(mrtydi_queries)} queries")

print("Loading Mr.TyDi qrels…")
mrtydi_qrels = mrtydi_loader.load_qrels()
print(f"  {len(mrtydi_qrels)} qrels entries")

# Corpus is shared (Arabic Wikipedia) — reuse the already-loaded corpus
evaluator_mrtydi = PipelineEvaluator(
    config, mrtydi_queries, corpus, mrtydi_qrels, dataset_name="mrtydi"
)

In [ ]:
# ── Zero-shot Scenario 1: Bi-encoder top-1000 on Mr.TyDi ──
bi_mrtydi = evaluator_mrtydi.evaluate_bi_encoder_top1000(
    bi_encoder=bi_model,
    model_name=model_label,
    kd_mode=kd_label,
)

# ── Zero-shot Scenario 2: Cross-encoder on BM25 Mr.TyDi candidates ──
# Load (or retrieve) a BM25 run for Mr.TyDi
bm25_mrtydi_path = os.path.join(config.data.cache_dir, "bm25_run_mrtydi.txt")
ce_bm25_mrtydi = None

if os.path.exists(bm25_mrtydi_path):
    bm25 = BM25Retriever()
    bm25_run_mrtydi = bm25.load_run(bm25_mrtydi_path, k=1000)
    ce_bm25_mrtydi = evaluator_mrtydi.evaluate_cross_encoder_on_bm25(
        cross_encoder=ce_reranker,
        bm25_run=bm25_run_mrtydi,
        model_name=model_label,
        kd_mode=kd_label,
    )
else:
    print("No BM25 run for Mr.TyDi — run BM25Retriever with Pyserini to get one.")
    print("  Topics : mrtydi-v1.1-arabic-test")
    print("  Index  : mrtydi-v1.1-ar")

zeroshot_results = [bi_mrtydi]
if ce_bm25_mrtydi:
    zeroshot_results.append(ce_bm25_mrtydi)

df_zeroshot = pd.DataFrame([r.as_dict() for r in zeroshot_results])
print("\n=== Zero-Shot (Mr.TyDi) ===")
display(df_zeroshot)

In [ ]:
# ── Compute Generalization Ratios ──
gr_results = []

# GR for bi-encoder (FAISS @1000)
gr_bi = PipelineEvaluator.compute_generalization_ratio(
    in_domain=bi_mmarco,
    zero_shot=bi_mrtydi,
)
gr_results.append(gr_bi)

# GR for cross-encoder on BM25 (if both in-domain and zero-shot runs are available)
if ce_bm25_mmarco and ce_bm25_mrtydi:
    gr_ce_bm25 = PipelineEvaluator.compute_generalization_ratio(
        in_domain=ce_bm25_mmarco,
        zero_shot=ce_bm25_mrtydi,
    )
    gr_results.append(gr_ce_bm25)

# Display GR table
df_gr = pd.DataFrame([g.as_dict() for g in gr_results])
print("\n=== Generalization Ratio ===")
display(df_gr)

# Log to W&B
for g in gr_results:
    wandb_logger.log_metrics(
        {f"GR/{g.stage}/{k}": v for k, v in g.gr.items()}
    )
wandb_logger.log_summary({f"best_{k}": v for g in gr_results for k, v in g.gr.items()})

## 11. Generate HTML Report

In [ ]:
from src.report.reporter import ReportGenerator

all_results = indomain_results + zeroshot_results

reporter = ReportGenerator(config, results=all_results, gr_results=gr_results)

# Standard metric charts
reporter.plot_metrics_bar("charts/mrr10_bar.png",      "MRR@10")
reporter.plot_metrics_bar("charts/ndcg10_bar.png",     "NDCG@10")
reporter.plot_metrics_bar("charts/recall1000_bar.png", "R@1000")

# GR charts
reporter.plot_gr_bar("charts/gr_mrr10.png",      "GR_MRR@10")
reporter.plot_gr_bar("charts/gr_ndcg10.png",     "GR_NDCG@10")
reporter.plot_gr_bar("charts/gr_recall1000.png", "GR_Recall@1000")

report_path = reporter.generate()

if config.training.wandb_log_artifacts and config.use_wandb:
    wandb_logger.log_artifact(report_path, "pipeline-report", "report", "HTML evaluation report")
    wandb_logger.log_artifact("charts/",   "metric-charts",  "report", "Metric and GR bar charts")

from IPython.display import IFrame
IFrame(report_path, width='100%', height=750)

## 12. Finish W&B Run

In [ ]:
wandb_logger.finish()
print("Done.")

## 13. (Optional) Multi-Model Evaluation + GR

Evaluates all pre-trained models and computes GR for each.

- **Bi-encoder**: FAISS @1000 on mMARCO (in-domain) and Mr.TyDi (zero-shot)
- **Cross-encoder**: reranking on BM25 candidates (both datasets)

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
from src.evaluation.evaluator import PipelineEvaluator
from src.evaluation.metrics import compute_generalization_ratio
from src.report.reporter import ReportGenerator
from src.retrieval.faiss_retrieval import FaissRetriever
from src.retrieval.bm25_retrieval import BM25Retriever

# (bi_encoder_id, cross_encoder_id, label, kd_mode)
model_pairs = [
    ("hatemestinbejaia/mmarco-Arabic-AraDPR-bi-encoder-NoKD-v1",
     "hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-NoKD-v1",
     "AraDPR", "nokd"),
    ("hatemestinbejaia/mmarco-Arabic-AraDPR-bi-encoder-KD-v1",
     "hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-KD-v1",
     "AraDPR", "pairwise_kd"),
    ("hatemestinbejaia/mmarco-Arabic-AraElectra-bi-encoder-KD-v1",
     "hatemestinbejaia/mmarco-Arabic-AraElectra-cross-encoder-KD-v1",
     "AraELECTRA", "pairwise_kd"),
    ("hatemestinbejaia/mmarco-Arabic-mMiniLML-bi-encoder-KD-v1",
     "hatemestinbejaia/mmarco-Arabic-mMiniLML-cross-encoder-KD-v1",
     "mMiniLML", "pairwise_kd"),
    ("hatemestinbejaia/mmarco-Arabic-mMiniLML-bi-encoder-KD-v1",
     "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
     "mMiniLMv2CE", "nokd"),
]

wb_run = wandb_logger.init(config)

# BM25 runs (load from cache if available)
bm25_mmarco_path = os.path.join(config.data.cache_dir, "bm25_run.txt")
bm25_mrtydi_path = os.path.join(config.data.cache_dir, "bm25_run_mrtydi.txt")
bm25 = BM25Retriever()
bm25_run_mmarco = bm25.load_run(bm25_mmarco_path) if os.path.exists(bm25_mmarco_path) else None
bm25_run_mrtydi = bm25.load_run(bm25_mrtydi_path) if os.path.exists(bm25_mrtydi_path) else None

all_results, all_gr = [], []

for bi_id, ce_id, label, kd_mode in model_pairs:
    print(f"\n{'='*60}\n{label} ({kd_mode})")

    bi  = SentenceTransformer(bi_id)
    ce  = CrossEncoder(ce_id, max_length=256)

    # ── In-domain (mMARCO) ──
    eval_mm = PipelineEvaluator(config, queries, corpus, qrels, dataset_name="mmarco")

    bi_mm = eval_mm.evaluate_bi_encoder_top1000(bi, label, kd_mode)
    print(f"  [mMARCO] bi@1000  MRR@10={bi_mm.mrr_at_10:.4f}  R@1000={bi_mm.recall_at_1000:.4f}")

    ce_bm25_mm = None
    if bm25_run_mmarco:
        ce_bm25_mm = eval_mm.evaluate_cross_encoder_on_bm25(ce, bm25_run_mmarco, label, kd_mode)
        print(f"  [mMARCO] CE@BM25  MRR@10={ce_bm25_mm.mrr_at_10:.4f}  NDCG@10={ce_bm25_mm.ndcg_at_10:.4f}")

    # ── Zero-shot (Mr.TyDi) ──
    eval_ty = PipelineEvaluator(config, mrtydi_queries, corpus, mrtydi_qrels, dataset_name="mrtydi")

    bi_ty = eval_ty.evaluate_bi_encoder_top1000(bi, label, kd_mode)
    print(f"  [Mr.TyDi] bi@1000  MRR@10={bi_ty.mrr_at_10:.4f}  R@1000={bi_ty.recall_at_1000:.4f}")

    ce_bm25_ty = None
    if bm25_run_mrtydi:
        ce_bm25_ty = eval_ty.evaluate_cross_encoder_on_bm25(ce, bm25_run_mrtydi, label, kd_mode)
        print(f"  [Mr.TyDi] CE@BM25  MRR@10={ce_bm25_ty.mrr_at_10:.4f}  NDCG@10={ce_bm25_ty.ndcg_at_10:.4f}")

    # ── Generalization Ratios ──
    gr_bi = PipelineEvaluator.compute_generalization_ratio(bi_mm, bi_ty)
    print(f"  GR bi@1000: {gr_bi.gr}")

    gr_ce = None
    if ce_bm25_mm and ce_bm25_ty:
        gr_ce = PipelineEvaluator.compute_generalization_ratio(ce_bm25_mm, ce_bm25_ty)
        print(f"  GR CE@BM25: {gr_ce.gr}")

    batch_results = [bi_mm, bi_ty]
    batch_gr = [gr_bi]
    if ce_bm25_mm: batch_results.append(ce_bm25_mm)
    if ce_bm25_ty: batch_results.append(ce_bm25_ty)
    if gr_ce:      batch_gr.append(gr_ce)

    all_results.extend(batch_results)
    all_gr.extend(batch_gr)

    wandb_logger.log_metrics({
        f"{label}/mmarco_bi_MRR@10":   bi_mm.mrr_at_10,
        f"{label}/mrtydi_bi_MRR@10":   bi_ty.mrr_at_10,
        f"{label}/GR_bi_MRR@10":       gr_bi.gr.get("GR_MRR@10", 0.0),
        **({f"{label}/mmarco_ce_MRR@10": ce_bm25_mm.mrr_at_10} if ce_bm25_mm else {}),
        **({f"{label}/mrtydi_ce_MRR@10": ce_bm25_ty.mrr_at_10} if ce_bm25_ty else {}),
        **({f"{label}/GR_ce_MRR@10": gr_ce.gr.get("GR_MRR@10", 0.0)} if gr_ce else {}),
    })

# Final report
reporter = ReportGenerator(config, results=all_results, gr_results=all_gr)
reporter.plot_metrics_bar("charts/all_bi_mrr10.png",      "MRR@10")
reporter.plot_metrics_bar("charts/all_bi_recall1000.png", "R@1000")
reporter.plot_gr_bar("charts/gr_bi_mrr10.png",      "GR_MRR@10")
reporter.plot_gr_bar("charts/gr_bi_recall1000.png", "GR_Recall@1000")
if any(r.stage == "reranking_bm25" for r in all_results):
    reporter.plot_gr_bar("charts/gr_ce_ndcg10.png",  "GR_NDCG@10")

report_path = reporter.generate()
wandb_logger.log_results(all_results, config)
if config.training.wandb_log_artifacts and config.use_wandb:
    wandb_logger.log_artifact(report_path, "multi-model-report", "report")

wandb_logger.finish()

df_all = pd.DataFrame([r.as_dict() for r in all_results])
df_gr  = pd.DataFrame([g.as_dict() for g in all_gr])
print("\n=== All Metrics ==="); display(df_all)
print("\n=== Generalization Ratios ==="); display(df_gr)